# Gold Layer Notebook

Build star schema (dimensions + fact) from Silver layer and store as Delta tables.

In [5]:
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, lit, dense_rank, year, quarter, month, weekofyear,
    dayofmonth, dayofweek, date_format, expr
)

In [6]:
MINIO_ENDPOINT = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET = "crypto-warehouse"

import sys
import subprocess

# Keep Spark and Delta versions compatible
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pyspark==4.0.0", "delta-spark==4.0.0"])

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Stop any existing session so new dependency/config set is applied
try:
    spark.stop()
except Exception:
    pass

builder = (
    SparkSession.builder.appName("DataWarehouse-ETL")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    # Delta Lake (explicitly required)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # MinIO (S3A)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

extra_packages = [
    "org.apache.hadoop:hadoop-aws:3.4.1",
    "software.amazon.awssdk:bundle:2.31.58",
]

spark = configure_spark_with_delta_pip(builder, extra_packages=extra_packages).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("spark.sql.extensions =", spark.conf.get("spark.sql.extensions", "<missing>"))
print("spark.sql.catalog.spark_catalog =", spark.conf.get("spark.sql.catalog.spark_catalog", "<missing>"))
spark

26/04/23 13:04:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/23 13:04:58 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/23 13:04:58 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension
spark.sql.catalog.spark_catalog = org.apache.spark.sql.delta.catalog.DeltaCatalog


In [7]:
BASE_URI = f"s3a://{MINIO_BUCKET}"
SILVER_PATH = f"{BASE_URI}/silver"
GOLD_BASE = f"{BASE_URI}/gold"

DIM_ASSET_PATH = f"{GOLD_BASE}/dim_asset"
DIM_DATE_PATH = f"{GOLD_BASE}/dim_date"
DIM_TIME_PATH = f"{GOLD_BASE}/dim_time"
FACT_PATH = f"{GOLD_BASE}/fact_daily_ohlc"

print("MINIO_ENDPOINT:", MINIO_ENDPOINT)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_BASE:", GOLD_BASE)

MINIO_ENDPOINT: http://localhost:9010
SILVER_PATH: s3a://crypto-warehouse/silver
GOLD_BASE: s3a://crypto-warehouse/gold


In [8]:
silver = spark.read.format("delta").load(SILVER_PATH)
print("Silver rows:", silver.count())
silver.select("asset_symbol", "trade_date").show(10, truncate=False)

Silver rows: 7045
+------------+----------+
|asset_symbol|trade_date|
+------------+----------+
|ETH         |2018-01-01|
|ETH         |2018-01-02|
|ETH         |2018-01-03|
|ETH         |2018-01-04|
|ETH         |2018-01-05|
|ETH         |2018-01-06|
|ETH         |2018-01-07|
|ETH         |2018-01-08|
|ETH         |2018-01-09|
|ETH         |2018-01-10|
+------------+----------+
only showing top 10 rows


In [10]:
dim_asset = (
    silver.select("asset_symbol").distinct()
    .withColumn("asset_key", dense_rank().over(Window.orderBy("asset_symbol")))
    .withColumn("asset_name",
        expr("CASE asset_symbol WHEN 'BTC' THEN 'Bitcoin' WHEN 'ETH' THEN 'Ethereum' WHEN 'LTC' THEN 'Litecoin' ELSE 'Unknown' END")
    )
    .withColumn("asset_category", lit("Layer1"))
    .withColumn("exchange", lit("Spot"))
    .withColumn("is_active", lit(True))
    .select("asset_key", "asset_symbol", "asset_name", "asset_category", "exchange", "is_active")
)

dim_asset.write.format("delta").mode("overwrite").save(DIM_ASSET_PATH)
dim_asset.show()

26/04/23 13:06:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 1

+---------+------------+----------+--------------+--------+---------+
|asset_key|asset_symbol|asset_name|asset_category|exchange|is_active|
+---------+------------+----------+--------------+--------+---------+
|        1|         BTC|   Bitcoin|        Layer1|    Spot|     true|
|        2|         ETH|  Ethereum|        Layer1|    Spot|     true|
|        3|         LTC|  Litecoin|        Layer1|    Spot|     true|
+---------+------------+----------+--------------+--------+---------+



26/04/23 13:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [11]:
date_bounds = silver.selectExpr("min(trade_date) as min_date", "max(trade_date) as max_date").collect()[0]
min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

dim_date = spark.sql(f"""
WITH date_spine AS (
  SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS full_date
)
SELECT
  CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
  full_date,
  year(full_date) AS year,
  quarter(full_date) AS quarter,
  month(full_date) AS month,
  date_format(full_date, 'MMMM') AS month_name,
  weekofyear(full_date) AS week_of_year,
  dayofmonth(full_date) AS day_of_month,
  dayofweek(full_date) AS day_of_week,
  date_format(full_date, 'EEEE') AS day_name,
  CASE WHEN dayofweek(full_date) IN (1, 7) THEN true ELSE false END AS is_weekend
FROM date_spine
""")

dim_date.write.format("delta").mode("overwrite").save(DIM_DATE_PATH)
dim_date.orderBy("full_date").show(10, truncate=False)

+--------+----------+----+-------+-----+----------+------------+------------+-----------+---------+----------+
|date_key|full_date |year|quarter|month|month_name|week_of_year|day_of_month|day_of_week|day_name |is_weekend|
+--------+----------+----+-------+-----+----------+------------+------------+-----------+---------+----------+
|20180101|2018-01-01|2018|1      |1    |January   |1           |1           |2          |Monday   |false     |
|20180102|2018-01-02|2018|1      |1    |January   |1           |2           |3          |Tuesday  |false     |
|20180103|2018-01-03|2018|1      |1    |January   |1           |3           |4          |Wednesday|false     |
|20180104|2018-01-04|2018|1      |1    |January   |1           |4           |5          |Thursday |false     |
|20180105|2018-01-05|2018|1      |1    |January   |1           |5           |6          |Friday   |false     |
|20180106|2018-01-06|2018|1      |1    |January   |1           |6           |7          |Saturday |true      |
|

In [12]:
dim_time = spark.sql("""
WITH time_spine AS (
  SELECT explode(sequence(0, 86399)) AS time_of_day
)
SELECT
  time_of_day AS time_key,
  int(time_of_day / 3600) AS hour,
  int((time_of_day % 3600) / 60) AS minute,
  int(time_of_day % 60) AS second,
  CASE WHEN int(time_of_day / 3600) < 12 THEN 'AM' ELSE 'PM' END AS am_pm
FROM time_spine
""")

dim_time.write.format("delta").mode("overwrite").save(DIM_TIME_PATH)
print("dim_time rows:", dim_time.count())

dim_time rows: 86400


In [14]:

fact = (
    silver.alias("s")
    .join(dim_asset.alias("a"), col("s.asset_symbol") == col("a.asset_symbol"), "left")
    .withColumn("date_key", (year(col("s.trade_date")) * 10000 + month(col("s.trade_date")) * 100 + dayofmonth(col("s.trade_date"))).cast("int"))
    .withColumn("trade_key", dense_rank().over(Window.orderBy(col("s.trade_date"), col("s.asset_symbol"))))
    .select(
        "trade_key",
        col("a.asset_key").alias("asset_key"),
        "date_key",
        col("s.open_price").alias("open_price"),
        col("s.high_price").alias("high_price"),
        col("s.low_price").alias("low_price"),
        col("s.close_price").alias("close_price"),
        # col("s.adj_close_price").alias("adj_close_price"),
        col("s.volume").alias("volume"),
        col("s.market_cap").alias("market_cap"),
        col("s.daily_return").alias("daily_return"),
        col("s.price_range").alias("price_range"),
        col("s.ingestion_ts").alias("loaded_timestamp"),
        col("s.source_file").alias("data_source")
    )
)

fact.write.format("delta").mode("overwrite").save(FACT_PATH)
print("fact_daily_ohlc rows:", fact.count())
fact.orderBy(col("date_key").desc(), col("asset_key")).show(20, truncate=False)

26/04/23 13:08:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 1

fact_daily_ohlc rows: 7045


26/04/23 13:08:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 13:08:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 1

+---------+---------+--------+-----------+-----------+-----------+-----------+------+---------------+------------+-----------+--------------------------------+--------------+
|trade_key|asset_key|date_key|open_price |high_price |low_price  |close_price|volume|market_cap     |daily_return|price_range|loaded_timestamp                |data_source   |
+---------+---------+--------+-----------+-----------+-----------+-----------+------+---------------+------------+-----------+--------------------------------+--------------+
|7045     |3        |20240731|71.69      |72.84      |70.11      |70.19      |NULL  |NULL           |-0.02092342 |2.73       |2026-04-23T05:08:56.526699+00:00|LITECOIN24.csv|
|7044     |3        |20240730|73.75      |74.48      |71.39      |71.69      |NULL  |NULL           |-0.0279322  |3.09       |2026-04-23T05:08:56.526699+00:00|LITECOIN24.csv|
|7043     |3        |20240729|71.13      |76.62      |71.09      |73.75      |NULL  |NULL           |0.03683397  |5.53       

In [15]:
from pathlib import Path
import shutil


def _resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "BatchProcessor").exists() and (candidate / "datasets").exists():
            return candidate
    return cwd


def export_delta_to_single_csv(table_name: str, delta_path: str, export_root: Path) -> None:
    temp_dir = export_root / f"_{table_name}_tmp"
    final_csv = export_root / f"{table_name}.csv"

    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    if final_csv.exists():
        final_csv.unlink()

    df = spark.read.format("delta").load(delta_path)
    (
        df.coalesce(1)
        .write.mode("overwrite")
        .option("header", "true")
        .csv(str(temp_dir))
    )

    part_files = list(temp_dir.glob("part-*.csv"))
    if not part_files:
        raise FileNotFoundError(f"No CSV part file generated for table: {table_name}")

    shutil.move(str(part_files[0]), str(final_csv))
    shutil.rmtree(temp_dir)
    print(f"Exported {table_name}: {final_csv}")


repo_root = _resolve_repo_root()
export_dir = repo_root / "dataset" / "gold"
export_dir.mkdir(parents=True, exist_ok=True)


GOLD_TABLE_PATHS = {
    "dim_asset": DIM_ASSET_PATH,
    "dim_date": DIM_DATE_PATH,
    "dim_time": DIM_TIME_PATH,
    "fact_daily_ohlc": FACT_PATH,
}

for table_name, table_path in GOLD_TABLE_PATHS.items():
    export_delta_to_single_csv(table_name, table_path, export_dir)

print(f"All Gold tables exported to: {export_dir}")

Exported dim_asset: /home/bnguyen/Desktop/finnhub-streaming-pipeline/dataset/gold/dim_asset.csv
Exported dim_date: /home/bnguyen/Desktop/finnhub-streaming-pipeline/dataset/gold/dim_date.csv
Exported dim_time: /home/bnguyen/Desktop/finnhub-streaming-pipeline/dataset/gold/dim_time.csv
Exported fact_daily_ohlc: /home/bnguyen/Desktop/finnhub-streaming-pipeline/dataset/gold/fact_daily_ohlc.csv
All Gold tables exported to: /home/bnguyen/Desktop/finnhub-streaming-pipeline/dataset/gold
